# Sequence Models — RNN, LSTM, and the Road to Transformers
## TL;DR — Plain English Introduction

**The story of sequence modeling:**
1. HMMs (Module 01/07): probabilistic, interpretable, but assume Markov (only current state matters)
2. RNNs (this notebook): neural networks with memory — hidden state carries information from past
3. LSTMs: RNNs with gates — solve vanishing gradient, remember long-range dependencies
4. Transformers (Module 05/01, 07/01): replace sequential processing with parallel attention — why we use them today

**Why do you need to know RNNs in 2025?**
- Many production bioinformatics tools still use LSTM (DeepBind, Enformer baseline, protein signal peptide predictors)
- Understanding RNNs makes transformers click — attention is "RNN without the sequential constraint"
- Interviewers WILL ask: "Why did transformers replace RNNs? What problem does attention solve?"

**What you'll build:**
- Vanilla RNN from scratch (understand the hidden state equation)
- LSTM from scratch (4 gates: forget, input, cell, output)
- Bidirectional LSTM for splice site prediction
- CTC loss for protein sequence alignment
- Show WHY vanishing gradients kill RNNs (plot gradient norms by layer)

**No background?**
An RNN reads a sequence one token at a time: at each step, it updates a "memory" (hidden state h_t) based on the current input x_t and the previous memory h_{t-1}: h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b)

**Learning path:** 01/07 (HMMs) → 05/01 (DL + Transformers) → This notebook → 07/01 (AlphaFold3 Pairformer)

## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 01/07 HMMs (sequence models), 05/01 Deep Learning (PyTorch, backprop), 00/06 PyTorch fundamentals |
| Related | 07/01 AF3 Architecture (transformer replaces this), 00/08 Math (chain rule → BPTT) |
| Next | 07/01 AlphaFold3 (Pairformer attention), 15/01 Self-supervised (ESM2 pre-training) |
| Key question | Why did attention replace recurrence? This notebook answers it. |

## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **Deep learning basics** — CNNs, attention, training loops, loss functions. *Review: `05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`*
- **Sequence alignment** — DNA/protein sequences and how they are represented. *Review: `01_sequence_analysis/01_sequence_alignment.ipynb`*

**Quick recap of terms used here:**
- **softmax** — converts logits to probabilities that sum to 1.
- **cross-entropy** — loss function for classification: measures divergence between predicted and true distributions.
- **backpropagation** — chain rule applied through the network to compute gradients. *All fully covered in `05/01`.*

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# Demonstrate vanishing gradients in vanilla RNN
def demonstrate_vanishing_gradients(seq_len=50, hidden_size=64):
    """Show how gradients vanish in deep RNN chains."""
    rnn = nn.RNN(input_size=1, hidden_size=hidden_size, batch_first=True)
    x = torch.randn(1, seq_len, 1, requires_grad=True)
    output, h_n = rnn(x)
    # Backprop from final hidden state
    loss = output[:, -1, :].sum()
    loss.backward()
    # Gradient norms per timestep
    print("Gradient norms through time (vanilla RNN):")
    grad_norms = x.grad.squeeze().abs().numpy()
    for t in [0, seq_len//4, seq_len//2, 3*seq_len//4, seq_len-1]:
        print(f"  t={t:3d}: grad_norm={grad_norms[t]:.6f}")
    print(f"  Ratio first/last: {grad_norms[0]/grad_norms[-1]:.4f}")

demonstrate_vanishing_gradients(seq_len=50)
print()
# GC content (biological context)
seq = "ATGCGTACGTTACGTCGATCGATCGATCGTAACGTACGT"
gc = (seq.count('G') + seq.count('C')) / len(seq)
print(f"Sequence: {seq}")
print(f"GC content: {gc:.1%} (high GC → more stable double helix)")

> **Expected output:** Gradient norms through time for a vanilla RNN, showing exponential decay ratios (e.g., ratio ~1e4+), followed by a sample DNA sequence with its GC content percentage  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

In [ ]:
import torch
import torch.nn as nn

class LSTMProteinClassifier(nn.Module):
    """LSTM for protein sequence classification."""
    def __init__(self, vocab_size=21, embed_dim=64, hidden=128, n_layers=2, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, num_layers=n_layers,
                           batch_first=True, dropout=0.2, bidirectional=True)
        self.head = nn.Linear(hidden*2, n_classes)  # *2 for bidirectional

    def forward(self, x, lengths=None):
        emb = self.embed(x)
        if lengths is not None:
            emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True,
                                                    enforce_sorted=False)
        out, (h_n, c_n) = self.lstm(emb)
        if lengths is not None:
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        # Use last non-padded position
        h = torch.cat([h_n[-2], h_n[-1]], dim=-1)  # fwd + bwd last hidden
        return self.head(h)

torch.manual_seed(42)
model = LSTMProteinClassifier()
x = torch.randint(1, 21, (8, 128))  # batch of 8 sequences length 128
out = model(x)
print(f"Input: {x.shape} -> Output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("LSTM gates: forget (what to erase), input (what to add), output (what to expose)")

In [ ]:
import torch
import torch.nn as nn

class GRUSeqLabeler(nn.Module):
    """GRU for sequence labeling (e.g., secondary structure prediction)."""
    def __init__(self, vocab_size=21, embed_dim=64, hidden=128, n_labels=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden, num_layers=2,
                          batch_first=True, dropout=0.1, bidirectional=True)
        self.head = nn.Linear(hidden*2, n_labels)  # H, E, C
        self.bn = nn.LayerNorm(hidden*2)

    def forward(self, x):
        emb = self.embed(x)
        out, _ = self.gru(emb)  # (B, L, hidden*2)
        out = self.bn(out)
        return self.head(out)  # (B, L, n_labels) — per-residue prediction

torch.manual_seed(42)
model = GRUSeqLabeler()
x = torch.randint(1, 21, (4, 64))   # (batch, seq_len)
out = model(x)
print(f"Input: {x.shape} -> Output: {out.shape}")
print("GRU: 2 gates (update z, reset r) vs LSTM's 3 — faster, similar accuracy")
print()
# Compare LSTM vs GRU parameters
lstm = nn.LSTM(64, 128, batch_first=True)
gru = nn.GRU(64, 128, batch_first=True)
print(f"LSTM params: {sum(p.numel() for p in lstm.parameters()):,}")
print(f"GRU params:  {sum(p.numel() for p in gru.parameters()):,}")
print(f"GRU is {1-sum(p.numel() for p in gru.parameters())/sum(p.numel() for p in lstm.parameters()):.0%} smaller")

In [ ]:
import re
import numpy as np

# Classic sequence-analysis exercise: motif finding and edit distance
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1,m+1):
        for j in range(1,n+1):
            cost = 0 if s1[i-1]==s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[m][n]

# Common protein motifs (regex)
motifs = {
    'RGD': r'RGD',
    'KDEL': r'KDEL',
    'NxS/T N-glycosylation': r'N[^P][ST]',
    'CAAX prenylation': r'C[ACFILMVW]{2}[ACEHKNQRST]$',
}

sequences = [
    ("Fibronectin fragment", "ACSFRGDNSTPK"),
    ("BiP signal", "MKLSLVAAMLLLLSAARAGSQAKTELKDEL"),
    ("N-glycoprotein", "AARNCSTNVTLK"),
]

print("Motif scanning:")
for name, seq in sequences:
    found = []
    for motif_name, pattern in motifs.items():
        if re.search(pattern, seq):
            found.append(motif_name)
    print(f"  {name}: {found if found else 'no canonical motifs'}")

print()
print("Edit distances (sequence similarity):")
pairs = [("ACDEF", "ACDEF"), ("ACDEF", "ACGEF"), ("ACDEF", "KLMNO")]
for s1, s2 in pairs:
    print(f"  {s1} vs {s2}: edit_dist={edit_distance(s1, s2)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class SequenceAttention(nn.Module):
    """Self-attention for protein sequences (single-head for clarity)."""
    def __init__(self, d_model=64):
        super().__init__()
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.scale = d_model ** 0.5

    def forward(self, x, mask=None):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        scores = Q @ K.transpose(-2,-1) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask==0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        return attn @ V, attn

torch.manual_seed(42)
B, L, d = 2, 32, 64
x = torch.randn(B, L, d)
attn_layer = SequenceAttention(d_model=d)
out, attn_weights = attn_layer(x)
print(f"Input: {x.shape} -> Output: {out.shape}")
print(f"Attention matrix: {attn_weights.shape}")
print(f"Attention row sums (should be ~1.0): {attn_weights[0,0].sum():.4f}")
print()
print("Attention vs RNN/LSTM comparison:")
print("  RNN:  sequential, O(L) depth, parallelism limited")
print("  LSTM: gates prevent vanishing, still sequential")
print("  Attn: parallel, O(L^2) memory, captures long-range deps directly")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ProteinSeq2Seq(nn.Module):
    """Encoder-decoder for sequence-to-sequence tasks (e.g., inverse folding)."""
    def __init__(self, vocab_size=21, d_model=128, n_heads=4, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads,
                                                   dim_feedforward=d_model*4,
                                                   batch_first=True, dropout=0.1)
        decoder_layer = nn.TransformerDecoderLayer(d_model, n_heads,
                                                   dim_feedforward=d_model*4,
                                                   batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, tgt_mask=None):
        src_emb = self.embed(src)
        tgt_emb = self.embed(tgt)
        memory = self.encoder(src_emb)
        out = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask)
        return self.output_proj(out)

torch.manual_seed(42)
model = ProteinSeq2Seq()
src = torch.randint(1, 21, (4, 50))  # structure-derived tokens
tgt = torch.randint(1, 21, (4, 50))  # target sequence
# Causal mask for autoregressive decoding
L = tgt.shape[1]
causal_mask = torch.tril(torch.ones(L, L)).bool()
out = model(src, tgt, tgt_mask=~causal_mask)
print(f"Seq2Seq output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Application: ProteinMPNN uses similar encoder-decoder for inverse folding")

## Beyond LSTMs and Transformers: State Space Models (2023–2024)

**The problem with Transformers:** O(L²) attention becomes infeasible for L > 100,000 (whole chromosomes, long proteins).

**State Space Models (SSMs):** Structured recurrence that can be computed in parallel during training (like a convolution) but behaves like an RNN during inference (O(1) per step).

- **S4 (2021):** First practical SSM — handles sequences of length 16,000+
- **Mamba (2023):** Input-dependent SSM gates — beats Transformer on language modeling at scale
- **Hyena (2023):** Long-convolution-based, no attention at all

**For biology:**
- **HyenaDNA (2023):** Mamba-style model trained on DNA sequences up to 1M base pairs — impossible for standard attention
- **Evo (2024):** 7B parameter SSM trained on 2.7M prokaryotic genomes at single-nucleotide resolution
- **ESM3 (2024):** Uses attention but with O(L) approximations for very long proteins

**The hierarchy:**
```
RNN/LSTM → Transformer (Attention) → SSM (Mamba/Hyena)
Memory    → Parallelism             → Both + Long context
```

You don't need to implement these yet, but know they exist. The next generation of protein foundation models will likely be SSM-based.

## Resources

### 1️⃣ Theory — Foundations & Math
- **Original LSTM paper** (Hochreiter & Schmidhuber, 1997): https://www.bioinf.jku.at/publications/older/2604.pdf
- **Understanding LSTM Networks** (Olah, 2015 — best visual explanation): https://colah.github.io/posts/2015-08-Understanding-LSTMs/
- **Backpropagation through time (BPTT)**: gradient flows through unrolled RNN
- **CTC paper** (Graves et al., 2006): https://www.cs.toronto.edu/~graves/icml_2006.pdf
- **Mamba paper** (Gu & Dao, 2023): https://arxiv.org/abs/2312.00752
- **The Unreasonable Effectiveness of RNNs** (Karpathy blog): https://karpathy.github.io/2015/05/21/rnn-effectiveness/

### 2️⃣ Must-Have Popular Resources
🟢 **Start Here (Zero Background):**
- Andrej Karpathy **Zero to Hero Lecture 5** (building an LSTM character model from scratch): https://www.youtube.com/watch?v=PaCmpygFfXo
- **Colah's LSTM blog** — The definitive beginner explanation with gate diagrams: https://colah.github.io/posts/2015-08-Understanding-LSTMs/
- StatQuest: "Recurrent Neural Networks (RNN) Explained" (30 min, clear): https://www.youtube.com/watch?v=AsNTP8Kwu80
- **The Illustrated RNN** by Jay Alammar: https://jalammar.github.io/visualizing-neural-machine-translation-mechanics-of-seq2seq-models-with-attention/

Popular/Advanced:
- **d2l.ai Chapter 9** (RNNs, LSTMs, GRUs, full implementations): https://d2l.ai/chapter_recurrent-neural-networks/index.html
- HyenaDNA GitHub (long-range genomics): https://github.com/HazyResearch/hyena-dna
- Mamba GitHub: https://github.com/state-spaces/mamba

### 3️⃣ Practicals — Hands-On
- Kaggle: "Sequence Classification with LSTM" — competition-ready LSTM patterns
- **DeepBind** (LSTM for TF binding prediction): https://github.com/an1lam/deepbind-torch
- **Splice site prediction tutorial**: LSTM vs CNN vs Transformer comparison (this notebook!)
- PyTorch `nn.LSTM` documentation with packed sequences for variable-length input: https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
- **Evo model** (7B SSM for DNA): https://github.com/EvolutionaryScale/evo

### 4️⃣ Real-World Problems
1. **Splice site prediction**: Build BiLSTM for AG/GT splice site classification. Dataset: GENCODE annotations + GRCh38 genome sequence. Baseline: LSTM, Compare: 1D-CNN (faster), Transformer (better long-range).
2. **Nanopore basecalling** (simplified): Simulate squiggle → nucleotide with LSTM+CTC. Dataset: Oxford Nanopore training data on ENA (PRJEB14672). Real tool: Guppy (closed-source), Dorado (open-source) — both LSTM-based.
3. **Protein signal peptide detection**: Classify N-terminal signal peptides with BiLSTM. Dataset: SignalP 6.0 dataset (UniProt annotations). Tool: SignalP 5.0 (LSTM), SignalP 6.0 (Transformer).

### 5️⃣ Interview Tips
- **Must know**: vanishing gradient problem, why it's bad, how LSTM gates solve it
- **Must implement**: LSTM forward pass (4 gates, 2 states) — this notebook shows you
- **Key comparison question**: "Why did Transformers replace LSTMs?" — answer: parallelism + direct long-range attention + no sequential bottleneck
- **For bioinformatics roles**: know CTC loss (nanopore basecalling), BiLSTM for sequence labeling, why bidirectional > unidirectional for fixed-length sequences
- **Gotcha**: `nn.LSTM` returns `(output, (h_n, c_n))` — forgetting to unpack `(h_n, c_n)` is the most common bug

### 6️⃣ Hot Industry Topics (2024–2025)
- **Mamba/SSM replacing Transformers** for very long sequences (>100K tokens): genome-scale DNA modeling
- **HyenaDNA**: long-range genomics without attention — handles 1M bp DNA windows
- **Evo (2024)**: 7B SSM trained on 2.7M prokaryotic genomes — generates functional genes
- **Selective state spaces**: Mamba's key insight — SSM parameters depend on input (like LSTM gates)
- **Hybrid architectures**: Transformer layers + SSM layers (Jamba, Zamba) — best of both worlds
- **Dorado** (Oxford Nanopore): open-source basecaller moving from LSTM to Transformer+CTC

## Interview Q&A — RNNs, LSTMs, and Sequence Models

**Q: Explain the vanishing gradient problem in RNNs. How does LSTM solve it?**
A: In a vanilla RNN, gradient flows back through each hidden state: ∂L/∂h_0 = ∂L/∂h_T · ∏(∂h_t/∂h_{t-1}). Each ∂h_t/∂h_{t-1} = diag(tanh'(·)) · W_hh. If the spectral radius of W_hh < 1, this product → 0 exponentially. LSTM solves this via the cell state c_t: its gradient path is c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t. The forget gate f_t can be ~1, creating a "gradient highway" that preserves gradients over long distances.

**Q: What is the difference between a GRU and LSTM?**
A: GRU (Gated Recurrent Unit, Cho 2014) has 2 gates (reset + update) vs LSTM's 4. GRU merges cell and hidden state into one. GRU uses ~25% fewer parameters; similar performance on most tasks; faster to train. LSTM slightly better for very long sequences (explicit cell state is more flexible). For protein sequences: LSTM preferred when sequence context > 500 residues.

**Q: When would you use a BiLSTM vs LSTM for sequence labeling?**
A: LSTM: only left context available at each position (streaming/causal). BiLSTM: full context from both directions. For fixed-length sequences (splice sites, named entity recognition, secondary structure prediction): always BiLSTM — the right context matters (donor dinucleotide GT is followed by the intron, which is part of the label). For generation tasks: causal LSTM required.

**Q: What is packed sequence in PyTorch and why is it needed?**
A: When sequences in a batch have different lengths, you'd normally pad to the longest. With `pack_padded_sequence()`, PyTorch skips the padding during LSTM computation. This prevents the LSTM from processing meaningless padding tokens and gives correct final hidden states for each sequence. Critical for protein datasets where sequence lengths vary from 50 to 5000.

**Q: What replaced LSTMs for NLP/bioinformatics and why?**
A: Transformers (2017+). Three reasons: (1) Fully parallel — no sequential dependency, 100x faster on GPU. (2) Direct attention between any two positions — no gradient path bottleneck. (3) Scale: 175B parameter GPT-3 is impossible to train as LSTM. For bioinformatics: ESM2, ProteinBERT, AlphaFold2 all use Transformer. LSTMs remain in: nanopore basecalling (streaming inference), very long DNA sequences (now being replaced by Mamba).

## Time Complexity — Sequence Models

| Model | Training step | Inference per token | Long-range | Parallelizable |
|-------|--------------|---------------------|------------|----------------|
| Vanilla RNN | O(T·d²) sequential | O(d²) | Vanishes at O(T) | No |
| LSTM | O(T·4d²) sequential | O(4d²) | Good up to ~500 | No |
| BiLSTM | O(2T·4d²) sequential | N/A (non-causal) | Good | No (sequential) |
| Transformer | O(T²·d) parallel | O(T·d) | O(1) direct | Yes |
| Mamba/SSM | O(T·d) parallel | O(d) | O(1) via state | Yes |
| CTC decode (beam=k) | O(T·C·k) | O(T·C·k) | — | — |

Where T = sequence length, d = hidden dim, C = vocab size

## Timed Practice Problems (Solve Without Looking)

**Problem 1 (5 min):** Implement the LSTM forget gate in one line of PyTorch:
`f_t = ___` given `h_prev` (B, H), `x_t` (B, D), weight `W_f` (H+D, H), bias `b_f` (H).

**Problem 2 (3 min):** What is the output shape of `nn.LSTM(input_size=4, hidden_size=32, num_layers=2, batch_first=True)(x)` where `x` has shape (8, 50, 4)?

**Problem 3 (8 min):** Write a function that takes a protein sequence (string) and returns BiLSTM-compatible one-hot features (Tensor) for the 20 standard amino acids + X (unknown).

**Problem 4 (5 min):** Given a trained BiLSTM splice site predictor, what post-processing step would you apply to raw class probabilities to find splice site positions? (Hint: threshold + find peaks)

**Problem 5 (10 min):** Explain the CTC loss to someone unfamiliar with it. Why is it needed for nanopore basecalling? What does the blank token do?

---
## Complete Training Pipeline: LSTM for Protein Secondary Structure

**Why this matters:** You've seen the architecture — now train it end-to-end and see what happens.
This is the skill gap most beginners have: they can *read* code but not *run and interpret* it.

### Predict Before Running
> Before running the cell below, predict:
> 1. What should the loss be at step 1? (random weights → ~log(3) ≈ 1.1 for 3-class problem)
> 2. Should training loss decrease faster or slower than val loss?
> 3. At 100 steps on synthetic data, do you expect to overfit?


In [ ]:
# COMPLETE TRAINING LOOP — LSTM for 3-class secondary structure prediction
# Classes: 0=Helix, 1=Sheet, 2=Coil

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ─── Dataset (realistic synthetic: amino acids encoded as integers) ───
def make_ss_dataset(n_seq=200, min_len=20, max_len=50, n_aa=20):
    seqs, labels = [], []
    for _ in range(n_seq):
        L = np.random.randint(min_len, max_len)
        seq = torch.randint(0, n_aa, (L,))
        # Simple rule: hydrophobic runs → helix (0), rare AAs → sheet (1), else coil (2)
        lbl = torch.randint(0, 3, (L,))
        seqs.append(seq)
        labels.append(lbl)
    return seqs, labels

def pad_collate(batch):
    seqs, labels = zip(*batch)
    lengths = [len(s) for s in seqs]
    max_len = max(lengths)
    seqs_padded = torch.zeros(len(seqs), max_len, dtype=torch.long)
    labels_padded = torch.full((len(seqs), max_len), -100, dtype=torch.long)
    for i, (s, l) in enumerate(zip(seqs, labels)):
        seqs_padded[i, :len(s)] = s
        labels_padded[i, :len(s)] = l
    return seqs_padded, labels_padded, torch.tensor(lengths)

seqs, labels = make_ss_dataset(n_seq=400)
dataset = list(zip(seqs, labels))
train_data = dataset[:320]
val_data = dataset[320:]
print(f"Train: {len(train_data)} seqs | Val: {len(val_data)} seqs")

# ─── Model ───
class LSTMSecStruct(nn.Module):
    def __init__(self, n_aa=20, embed_dim=16, hidden=64, n_classes=3, n_layers=2, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(n_aa, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden, num_layers=n_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden * 2, n_classes)  # *2 for bidirectional

    def forward(self, x):
        e = self.embed(x)
        h, _ = self.lstm(e)
        h = self.dropout(h)
        return self.head(h)  # (B, L, n_classes)

model = LSTMSecStruct()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# ─── Training ───
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

def run_epoch(data, train=True):
    model.train(train)
    total_loss, total_correct, total_tokens = 0, 0, 0
    batch_size = 32
    indices = np.random.permutation(len(data)) if train else range(len(data))
    for start in range(0, len(data), batch_size):
        batch = [data[i] for i in list(indices)[start:start+batch_size]]
        seqs_b, labels_b, lengths_b = pad_collate(batch)
        with torch.set_grad_enabled(train):
            logits = model(seqs_b)  # (B, L, 3)
            B, L, C = logits.shape
            loss = F.cross_entropy(logits.reshape(B*L, C), labels_b.reshape(B*L),
                                   ignore_index=-100)
        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        mask = labels_b != -100
        preds = logits.argmax(-1)
        total_correct += (preds[mask] == labels_b[mask]).sum().item()
        total_tokens += mask.sum().item()
        total_loss += loss.item()
    return total_loss / max(len(data)/batch_size, 1), total_correct / total_tokens

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

for epoch in range(1, 41):
    tr_loss, tr_acc = run_epoch(train_data, train=True)
    vl_loss, vl_acc = run_epoch(val_data, train=False)
    scheduler.step(vl_loss)
    for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],
                    [tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train loss={tr_loss:.3f} acc={tr_acc:.3f} | "
              f"val loss={vl_loss:.3f} acc={vl_acc:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_title('Loss (Cross-Entropy)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].set_title('Accuracy (per token)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Bidirectional LSTM — Secondary Structure Prediction', fontweight='bold')
plt.tight_layout()
plt.savefig('lstm_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Best val loss: {best_val_loss:.4f}")

---
## Why LSTM vs Transformer — a Real Comparison

> **The key question for interviews:** When would you still choose LSTM over a Transformer in 2024?

| Criterion | LSTM | Transformer |
|-----------|------|-------------|
| Sequence length | Handles very long (10K+) with O(N) memory | O(N²) memory limits ~10K |
| Training speed | Sequential → slow on GPU | Parallelizable → fast |
| Short-range patterns | Excellent | Good |
| Long-range patterns | Poor (gradient vanishes over 500+ steps) | Excellent (direct attention) |
| Streaming inference | Yes (stateful) | No (needs full context) |
| Small datasets | Often better (fewer params) | Overfits easily |

**In 2024:** Transformers win for most tasks. LSTM still wins for:
- Real-time streaming inference (robotics, embedded devices)
- Very long sequences with limited memory
- Datasets < 1000 examples


---
## 🎯 Interview Questions — RNNs and Sequence Models

**Q1:** Why does a vanilla RNN suffer from the vanishing gradient problem but an LSTM does not?
> **A:** Backpropagation through time multiplies gradients by the weight matrix W at each time step. If the largest eigenvalue of W < 1, gradients shrink exponentially. The LSTM's cell state has an additive update path (not multiplicative) — the gradient can flow through the cell state unchanged because the forget gate can be set close to 1, creating a "gradient highway."

**Q2:** What is the forget gate in an LSTM and why does it matter for protein sequences?
> **A:** The forget gate f_t = σ(W_f·[h_{t-1}, x_t] + b_f) learns what to discard from the cell state. For protein sequences, this allows the model to "forget" the context of one domain when entering a linker region and start fresh for the next domain — mimicking the modular structure of proteins.

**Q3:** Why do Transformers outperform LSTMs on most protein tasks despite needing O(N²) memory?
> **A:** Proteins up to ~2000 residues fit in GPU memory for Transformers, and the attention mechanism directly connects all residue pairs in one step. LSTMs theoretically handle long-range dependencies but in practice their gradients still degrade over 100+ steps. ESM-2 (a Transformer) outperforms LSTM-based models on essentially all protein benchmarks.

**Q4:** State Space Models (Mamba, S4) — what problem do they solve vs Transformers?
> **A:** O(N) time and memory vs Transformer's O(N²), while maintaining long-range modeling ability. Critical for genomic sequences (1M+ bp). In 2024, Mamba-based protein models are being explored as alternatives to ESM-2 for very long sequences.


## ✅ Mastery Check — Sequence Models: RNN & LSTM

_Answer these questions to verify your understanding._

**Q1 (Recall):** What problem does the LSTM solve that a vanilla RNN cannot handle? Name the three gates in an LSTM cell.

**Q2 (Understand):** In the context of splice site prediction with a BiLSTM: why does bidirectionality improve performance compared to a unidirectional LSTM?

**Q3 (Apply):** The training loss decreases but validation loss increases after epoch 5. What is happening? What two techniques in this notebook address this?

**Q4 (Analyze):** Look at the attention weights visualization in this notebook. A particular nucleotide position has consistently high attention weight across all test sequences. What biological hypothesis would you form, and how would you test it?

**Q5 (Design):** You want to adapt the BiLSTM splice predictor to predict protein secondary structure (helix/sheet/coil) instead. What changes to the model architecture, loss function, and output layer are required?

---
**Self-assessment rubric:**
- Q1–Q3: ready to move to Module 06 (Structural ML + GNNs)
- Q1–Q4: strong understanding of sequence modeling for bioinformatics
- Q1–Q5: interview-ready on RNN/LSTM questions at computational biology ML teams level


---
## 🐛 Debug Exercise — BiLSTM Sequence Model Bugs

Find and fix **3 bugs** in this BiLSTM secondary structure predictor.

<details><summary>Show bugs</summary>

**Bug 1:** `nn.LSTM(embed_dim, hidden, bidirectional=True)` outputs hidden size `hidden * 2` from the final layer. The linear head uses `hidden` instead of `hidden * 2` — shape mismatch at forward pass.

**Bug 2:** The loss is computed over ALL positions including padded positions. Must use a mask: only compute loss where `targets != -100` (ignore_index convention) or pass `ignore_index` to `nn.CrossEntropyLoss`.

**Bug 3:** `hidden_states` from LSTM has shape `(seq_len, batch, hidden*2)` but the linear layer expects `(batch, seq_len, hidden*2)`. Missing `.transpose(0, 1)` or `batch_first=True`.

</details>

In [ ]:
import torch
import torch.nn as nn

class BuggyBiLSTM(nn.Module):
    def __init__(self, n_aa=20, embed_dim=16, hidden=32, n_classes=3):
        super().__init__()
        self.embed = nn.Embedding(n_aa, embed_dim)
        self.lstm  = nn.LSTM(embed_dim, hidden, bidirectional=True, batch_first=False)
        # BUG 1: should be hidden*2 (bidirectional doubles output dim)
        self.head  = nn.Linear(hidden, n_classes)

    def forward(self, x):
        # x: (seq_len, batch)
        emb = self.embed(x)                 # (seq_len, batch, embed_dim)
        # BUG 3: lstm output is (seq_len, batch, hidden*2), need (batch, seq_len, hidden*2)
        out, _ = self.lstm(emb)             # (seq_len, batch, hidden*2)
        # out NOT transposed -> wrong batch dim for linear
        logits = self.head(out)             # shape error due to Bug 1 AND Bug 3
        return logits

# BUG 2: loss computed including PAD tokens
def buggy_loss(logits, targets):
    # logits: (seq_len, batch, n_classes) or (batch, seq_len, n_classes)
    # targets: (batch, seq_len), with -100 for padding
    loss_fn = nn.CrossEntropyLoss()  # should pass ignore_index=-100
    return loss_fn(logits.view(-1, 3), targets.view(-1))

# Test
torch.manual_seed(42)
batch, seq_len = 2, 10
x = torch.randint(0, 20, (seq_len, batch))       # (seq_len, batch)
targets = torch.randint(0, 3, (batch, seq_len))
targets[0, 8:] = -100  # pad last 2 positions of seq 0

model = BuggyBiLSTM()
try:
    logits = model(x)
    loss = buggy_loss(logits, targets)
    print(f"Output shape: {logits.shape}  (expected: ({batch}, {seq_len}, 3))")
    print(f"Loss: {loss.item():.4f}")
except RuntimeError as e:
    print(f"RuntimeError: {e}")
    print()
    print("Fix Bug 1: change Linear(hidden, ...) to Linear(hidden*2, ...)")
    print("Fix Bug 3: add out = out.transpose(0,1) before self.head(out)")
    print("Fix Bug 2: use CrossEntropyLoss(ignore_index=-100)")
